# IntruShield — Network Intrusion Detection on NSL-KDD
### Complete pipeline notebook (WebForge AI · PS 2: Intelligent Network Intrusion Detection System)

This notebook contains **the complete IntruShield code** and runs the whole pipeline end to end:

| Step | Module | What it does |
|---|---|---|
| 1 | `config.py` | paths, the 41 NSL-KDD feature names, attack → family mapping |
| 2 | `data.py` | downloads NSL-KDD, labels every row, runs the **data audit** |
| 3 | `features.py` | preprocessing pipeline (one-hot, log1p, scaling) fitted on training data only |
| 4 | `train.py` | trains & tunes 4 models for 2 tasks, selects the best, 5-fold CV, threshold, refit |
| 5 | `evaluate.py` | Protocol A (validation) vs Protocol B (KDDTest+ / KDDTest-21), all figures, SHAP |
| 6 | `report.py` | builds the self-contained `reports/eval_report.html` |
| 7 | `predict.py` | live demo: scores 10 test connections with SHAP reasons |

**Tasks:** binary (normal vs attack) and 5-class (Normal / DoS / Probe / R2L / U2R).

**Leakage rules:** preprocessing, tuning, model selection and the decision threshold use only KDDTrain+
(85 / 15 stratified split). KDDTest+ is read for the first time in the evaluation step. `random_state = 42` everywhere.

**How to run:** put this notebook in the project folder (e.g. `D:\IntruShield`), select the project's `.venv`
as the kernel (install once with `pip install ipykernel notebook` if needed), then **Run All**.
Each `%%writefile` cell writes a module to `src/`; each `%run` cell executes it.
Full run ≈ 8–10 min on a 16-core CPU (training is the long step).

## 0. Setup

In [ ]:
import os, sys, importlib
from pathlib import Path

ROOT = Path.cwd()
(ROOT / "src").mkdir(exist_ok=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Check the required libraries are installed
for pkg in ["pandas", "numpy", "sklearn", "xgboost", "lightgbm", "shap", "matplotlib", "seaborn", "joblib"]:
    m = importlib.import_module(pkg)
    print(f"{pkg:<12}{getattr(m, '__version__', '')}")
print("\nProject root:", ROOT, "| CPU cores:", os.cpu_count())

In [ ]:
%%writefile requirements.txt
pandas>=2.2
numpy>=1.26,<3
scikit-learn>=1.5
xgboost>=2.1
lightgbm>=4.5
shap>=0.46
matplotlib>=3.9
seaborn>=0.13
joblib>=1.4

## 1. Configuration — `config.py`
Shared settings used by every step:
- **41 features** per connection (basic, content, time-based traffic, host-based traffic) + attack label + difficulty.
- **Attack taxonomy:** every attack name (including 17 that appear only in the test set) mapped to its family —
  **DoS** (flooding), **Probe** (scanning), **R2L** (remote-to-local login attacks), **U2R** (privilege escalation).
- Skewed byte/count columns that get a `log1p` transform.

In [ ]:
%%writefile src/config.py
"""IntruShield - shared configuration: paths, column names, attack taxonomy."""
from pathlib import Path

RANDOM_STATE = 42
ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
CANDIDATES_DIR = MODELS_DIR / "candidates"
REPORTS_DIR = ROOT / "reports"
FIG_DIR = REPORTS_DIR / "figures"
for d in (DATA_DIR, MODELS_DIR, CANDIDATES_DIR, REPORTS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

RAW_BASE = "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/"
FILES = {"train": "KDDTrain+.txt", "test": "KDDTest+.txt", "test21": "KDDTest-21.txt"}

COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "attack", "difficulty",
]
FEATURES = COLUMNS[:41]
CATEGORICAL = ["protocol_type", "service", "flag"]
NUMERIC = [c for c in FEATURES if c not in CATEGORICAL]
# Heavily right-skewed byte / count columns -> log1p
SKEWED = [
    "duration", "src_bytes", "dst_bytes", "hot", "num_failed_logins", "num_compromised",
    "num_root", "num_file_creations", "num_shells", "num_access_files", "count", "srv_count",
    "dst_host_count", "dst_host_srv_count", "wrong_fragment", "urgent",
]

ATTACK_MAP = {
    "DoS": ["back", "land", "neptune", "pod", "smurf", "teardrop", "apache2", "mailbomb",
            "processtable", "udpstorm"],
    "Probe": ["ipsweep", "nmap", "portsweep", "satan", "mscan", "saint"],
    "R2L": ["ftp_write", "guess_passwd", "imap", "multihop", "phf", "spy", "warezclient",
            "warezmaster", "named", "sendmail", "snmpgetattack", "snmpguess", "xlock", "xsnoop",
            "worm"],
    "U2R": ["buffer_overflow", "loadmodule", "perl", "rootkit", "httptunnel", "ps", "sqlattack",
            "xterm"],
    "Normal": ["normal"],
}
LABEL_TO_CLASS = {a: c for c, lst in ATTACK_MAP.items() for a in lst}
CLASSES = ["Normal", "DoS", "Probe", "R2L", "U2R"]

NAVY, TEAL = "#0b2545", "#13a4a4"
PALETTE = ["#2a5ea8", "#0a9c9c", "#d9781c", "#7a5cc7", "#b83a3a"]  # CVD-validated

## 2. Data download, labelling and audit — `data.py`
Downloads `KDDTrain+`, `KDDTest+`, `KDDTest-21` (skipped if already in `data/`), maps every attack to its family,
and audits the data **before** any modelling: duplicates, class balance, constant columns and —
most importantly — **attack types that exist in the test set but never in training**.

In [ ]:
%%writefile src/data.py
"""IntruShield - data download, loading, labelling and audit."""
import json
import sys
import urllib.request

import pandas as pd

from config import (ATTACK_MAP, COLUMNS, DATA_DIR, FEATURES, FILES, LABEL_TO_CLASS, RAW_BASE,
                    REPORTS_DIR)


def download() -> None:
    for fname in FILES.values():
        dest = DATA_DIR / fname
        if dest.exists() and dest.stat().st_size > 0:
            continue
        url = RAW_BASE + fname
        print(f"Downloading {url}")
        try:
            urllib.request.urlretrieve(url, dest)
        except Exception as exc:  # noqa: BLE001
            sys.exit(f"Download of {fname} failed ({exc}). Place KDDTrain+.txt, KDDTest+.txt and "
                     f"KDDTest-21.txt manually in {DATA_DIR} and re-run.")


def load_split(name: str) -> pd.DataFrame:
    """Return a DataFrame with 41 features + attack, difficulty, label5, is_attack."""
    df = pd.read_csv(DATA_DIR / FILES[name], header=None, names=COLUMNS)
    df["attack"] = df["attack"].str.strip().str.rstrip(".")
    unmapped = sorted(set(df["attack"]) - set(LABEL_TO_CLASS))
    if unmapped:
        sys.exit(f"Unmapped attack labels in {FILES[name]}: {unmapped}. Extend ATTACK_MAP in "
                 f"src/config.py.")
    df["label5"] = df["attack"].map(LABEL_TO_CLASS)
    df["is_attack"] = (df["label5"] != "Normal").astype(int)
    return df


def audit(train: pd.DataFrame, test: pd.DataFrame, test21: pd.DataFrame) -> dict:
    train_attacks, test_attacks = set(train["attack"]), set(test["attack"])
    const_cols = [c for c in FEATURES if train[c].nunique() == 1]
    unseen = sorted(test_attacks - train_attacks)
    info = {
        "rows": {"train": len(train), "test": len(test), "test21": len(test21)},
        "duplicate_rows": {
            "train": int(train[FEATURES + ["attack"]].duplicated().sum()),
            "test": int(test[FEATURES + ["attack"]].duplicated().sum()),
            "test_rows_also_in_train": int(
                test[FEATURES].merge(train[FEATURES].drop_duplicates(), how="inner").shape[0]),
        },
        "class_distribution": {
            "train": train["label5"].value_counts().to_dict(),
            "test": test["label5"].value_counts().to_dict(),
            "test21": test21["label5"].value_counts().to_dict(),
        },
        "binary_distribution": {
            "train": train["is_attack"].value_counts().rename({0: "normal", 1: "attack"}).to_dict(),
            "test": test["is_attack"].value_counts().rename({0: "normal", 1: "attack"}).to_dict(),
        },
        "n_attack_types": {"train": len(train_attacks), "test": len(test_attacks)},
        "constant_columns": const_cols,
        "attacks_in_test_not_in_train": {
            a: {"class": LABEL_TO_CLASS[a], "test_rows": int((test["attack"] == a).sum())}
            for a in unseen},
        "unseen_attack_rows_in_test": int(test["attack"].isin(unseen).sum()),
        "attack_map": ATTACK_MAP,
        "difficulty_mean": {"train": float(train["difficulty"].mean()),
                            "test": float(test["difficulty"].mean())},
    }
    (REPORTS_DIR / "audit.json").write_text(json.dumps(info, indent=2))
    return info


if __name__ == "__main__":
    download()
    tr, te, te21 = load_split("train"), load_split("test"), load_split("test21")
    a = audit(tr, te, te21)
    print(json.dumps({k: v for k, v in a.items() if k != "attack_map"}, indent=2))
    print(f"Audit written to {REPORTS_DIR / 'audit.json'}")

In [ ]:
%run src/data.py

In [ ]:
import json
import pandas as pd

audit = json.loads((ROOT / "reports" / "audit.json").read_text())
print("Rows:", audit["rows"])
print("Distinct attack types -> train:", audit["n_attack_types"]["train"], "| test:", audit["n_attack_types"]["test"])
print("Test attack rows of types never seen in training:", audit["unseen_attack_rows_in_test"])
display(pd.DataFrame(audit["class_distribution"]).fillna(0).astype(int).rename_axis("class"))
display(pd.DataFrame(audit["attacks_in_test_not_in_train"]).T.sort_values("test_rows", ascending=False)
        .rename_axis("unseen attack type"))

## 3. Preprocessing — `features.py`
One scikit-learn `ColumnTransformer`, always fitted on training rows only:
- **One-hot** for `protocol_type`, `service`, `flag` (`handle_unknown="ignore"` so test-only services do not crash).
- **log1p** for heavily skewed byte / count columns.
- **StandardScaler** only for the linear model (trees do not need it).
- Constant columns (e.g. `num_outbound_cmds`) are dropped.

In [ ]:
%%writefile src/features.py
"""IntruShield - preprocessing pipeline (fit on training data only)."""
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from config import CATEGORICAL, NUMERIC, SKEWED


def build_preprocessor(constant_cols: list[str], scale: bool) -> ColumnTransformer:
    """OneHot for categoricals, log1p for skewed counts, optional StandardScaler (linear models).

    constant_cols are dropped. All statistics (categories, scaler mean/std) are learned in .fit,
    which must only ever see KDDTrain+ rows.
    """
    skewed = [c for c in SKEWED if c not in constant_cols]
    plain = [c for c in NUMERIC if c not in constant_cols and c not in skewed]

    def numeric_steps(log: bool):
        steps = []
        if log:
            steps.append(("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")))
        if scale:
            steps.append(("scale", StandardScaler()))
        return Pipeline(steps) if steps else "passthrough"

    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32),
             CATEGORICAL),
            ("log", numeric_steps(log=True), skewed),
            ("num", numeric_steps(log=False), plain),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

## 4. Training, tuning and model selection — `train.py`
For **both** tasks (binary and 5-class):
1. Stratified **85 / 15** split of KDDTrain+ (validation = 15 %), plus a duplicate-leakage check between the halves.
2. Four models: **Logistic Regression** (baseline), **Random Forest**, **XGBoost**, **LightGBM**, all class-balanced.
3. XGBoost & LightGBM tuned with `RandomizedSearchCV` (12 configs × 3-fold, macro-F1). Learning rate is capped and
   L2 regularisation added — higher rates made multiclass boosting diverge on some folds.
4. Best model selected by **validation macro-F1**, then **5-fold CV** on all of KDDTrain+ for stability.
5. Binary **decision threshold** chosen on validation only: maximum attack recall with false-alarm rate ≤ 5 %.
6. Best model refit on 100 % of KDDTrain+ and saved to `models/`.

In [ ]:
%%writefile src/train.py
"""IntruShield - train, tune, select and persist binary + multiclass models.

Protocol: stratified 85/15 split of KDDTrain+; all tuning / selection / thresholding on the 15%
validation split; KDDTest+ is never touched here.
"""
import json
import os
import time
import warnings

import joblib
import numpy as np
from lightgbm import LGBMClassifier
from scipy.stats import randint, uniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_curve
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

from config import CANDIDATES_DIR, CLASSES, FEATURES, MODELS_DIR, RANDOM_STATE, REPORTS_DIR
from data import load_split
from features import build_preprocessor

warnings.filterwarnings("ignore")
N_CPU = os.cpu_count() or 4
MAX_FAR = 0.05  # max false-alarm rate allowed when choosing the binary threshold


def make_estimators(task: str, n_jobs: int) -> dict:
    """Base (untuned) estimators. task in {'binary','multiclass'}."""
    n_cls = 2 if task == "binary" else len(CLASSES)
    return {
        "LogisticRegression": LogisticRegression(class_weight="balanced", max_iter=2000, C=1.0,
                                                 random_state=RANDOM_STATE),
        "RandomForest": RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample",
                                               n_jobs=n_jobs, random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                 tree_method="hist", n_jobs=n_jobs, random_state=RANDOM_STATE,
                                 objective="binary:logistic" if n_cls == 2 else "multi:softprob",
                                 eval_metric="logloss" if n_cls == 2 else "mlogloss"),
        "LightGBM": LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.1,
                                   class_weight="balanced", n_jobs=n_jobs, verbose=-1,
                                   random_state=RANDOM_STATE),
    }


# Learning-rate ranges are capped: with class-balanced weights (U2R weight ~500x) a boosting rate above
# ~0.2 makes multiclass LightGBM/XGBoost diverge on some folds (observed macro-F1 0.28-0.96 across folds).
SEARCH_SPACES = {
    "XGBoost": {
        "clf__n_estimators": randint(100, 300),
        "clf__max_depth": randint(4, 9),
        "clf__learning_rate": uniform(0.03, 0.17),
        "clf__subsample": uniform(0.6, 0.4),
        "clf__colsample_bytree": uniform(0.5, 0.5),
        "clf__min_child_weight": randint(1, 6),
        "clf__reg_lambda": uniform(0.5, 4.5),
    },
    "LightGBM": {
        "clf__n_estimators": randint(100, 400),
        "clf__num_leaves": randint(15, 96),
        "clf__learning_rate": uniform(0.03, 0.12),
        "clf__subsample": uniform(0.6, 0.4),
        "clf__subsample_freq": [1],
        "clf__colsample_bytree": uniform(0.5, 0.5),
        "clf__min_child_samples": randint(20, 80),
        "clf__reg_lambda": uniform(0.5, 4.5),
    },
}


def make_pipeline(name: str, est, constant_cols):
    return Pipeline([("prep", build_preprocessor(constant_cols, scale=name == "LogisticRegression")),
                     ("clf", est)])


def fit_kwargs(name: str, y):
    """XGBoost gets balanced sample weights; the others use class_weight."""
    if name == "XGBoost":
        return {"clf__sample_weight": compute_sample_weight("balanced", y)}
    return {}


def choose_threshold(y_true, p_attack, max_far=MAX_FAR):
    """Maximise attack recall subject to validation FPR <= max_far.

    Recall is monotone non-increasing in the threshold, so the optimum is the LOWEST score threshold
    whose false-alarm rate still respects the budget. When validation is (nearly) separable this
    deliberately spends the whole false-alarm budget on sensitivity - the a-priori choice for a
    detector that must face attack types absent from validation.
    """
    fpr, tpr, thr = roc_curve(y_true, p_attack, drop_intermediate=False)
    ok = np.where(fpr <= max_far)[0]
    best = ok[-1]  # fpr/tpr are non-decreasing along the curve -> last feasible index = max recall
    t = float(min(thr[best], 1.0))
    grid = np.unique(np.concatenate([np.logspace(-7, 0, 120), [0.5]]))
    curve = []
    for cut in grid:
        pred = (p_attack >= cut).astype(int)
        curve.append({"threshold": float(cut),
                      "recall": float(pred[y_true == 1].mean()),
                      "false_alarm_rate": float(pred[y_true == 0].mean())})
    return t, float(tpr[best]), float(fpr[best]), curve


def run_task(task, X_tr, y_tr, X_val, y_val, constant_cols, log):
    results, fitted = {}, {}
    base = make_estimators(task, n_jobs=N_CPU)
    cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    for name, est in base.items():
        best_params = {}
        if name in SEARCH_SPACES:
            # single-threaded estimator clones inside a fully parallel search (no oversubscription)
            search_est = make_estimators(task, n_jobs=1)[name]
            search = RandomizedSearchCV(
                make_pipeline(name, search_est, constant_cols), SEARCH_SPACES[name], n_iter=12,
                cv=cv3, scoring="f1_macro", n_jobs=-1, random_state=RANDOM_STATE, refit=False)
            t0 = time.perf_counter()
            search.fit(X_tr, y_tr, **fit_kwargs(name, y_tr))
            best_params = {k.replace("clf__", ""): (v.item() if hasattr(v, "item") else v)
                           for k, v in search.best_params_.items()}
            est.set_params(**best_params)
            log(f"  [{task}] {name} tuned in {time.perf_counter() - t0:.0f}s, "
                f"cv f1_macro={search.best_score_:.4f} params={best_params}")
        pipe = make_pipeline(name, est, constant_cols)
        t0 = time.perf_counter()
        pipe.fit(X_tr, y_tr, **fit_kwargs(name, y_tr))
        train_time = time.perf_counter() - t0
        f1 = f1_score(y_val, pipe.predict(X_val), average="macro")
        log(f"  [{task}] {name}: val macro-F1={f1:.4f}  train={train_time:.1f}s")
        results[name] = {"val_f1_macro": float(f1), "train_time_s": float(train_time),
                         "best_params": best_params}
        fitted[name] = pipe
        joblib.dump(pipe, CANDIDATES_DIR / f"{task}_{name}.joblib")
    best = max(results, key=lambda n: results[n]["val_f1_macro"])
    log(f"  [{task}] BEST = {best}")
    return best, results, fitted


def main():
    t_start = time.perf_counter()
    lines = []

    def log(msg):
        print(msg, flush=True)
        lines.append(msg)

    train = load_split("train")
    constant_cols = [c for c in FEATURES if train[c].nunique() == 1]
    log(f"Dropping constant columns: {constant_cols}")
    X, y5, yb = train[FEATURES], train["label5"], train["is_attack"]
    # One stratified split shared by both tasks (stratify on the 5-class label).
    idx_tr, idx_val = train_test_split(np.arange(len(train)), test_size=0.15,
                                       stratify=y5, random_state=RANDOM_STATE)
    X_tr, X_val = X.iloc[idx_tr], X.iloc[idx_val]
    le = LabelEncoder().fit(CLASSES)
    y5_tr, y5_val = le.transform(y5.iloc[idx_tr]), le.transform(y5.iloc[idx_val])
    yb_tr, yb_val = yb.iloc[idx_tr].values, yb.iloc[idx_val].values
    log(f"Train/val split: {len(idx_tr)} / {len(idx_val)} rows")

    # Duplicate-leakage check between the two halves of the split (feature vectors only)
    dup_val_in_tr = int(X_val.merge(X_tr.drop_duplicates(), how="inner").shape[0])
    log(f"Validation rows whose feature vector also appears in train split: {dup_val_in_tr}")

    summary = {"split": {"train_rows": int(len(idx_tr)), "val_rows": int(len(idx_val)),
                         "val_rows_duplicated_in_train": dup_val_in_tr},
               "constant_cols": constant_cols, "classes": CLASSES, "tasks": {}}

    for task, (ytr, yval) in {"binary": (yb_tr, yb_val), "multiclass": (y5_tr, y5_val)}.items():
        log(f"\n=== {task.upper()} ===")
        best, results, fitted = run_task(task, X_tr, ytr, X_val, yval, constant_cols, log)
        best_pipe = fitted[best]

        # 5-fold stratified CV of the best configuration on the full KDDTrain+
        y_full = yb.values if task == "binary" else le.transform(y5)
        cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        t0 = time.perf_counter()
        scores = cross_val_score(best_pipe, X, y_full, cv=cv5, scoring="f1_macro",
                                 n_jobs=1, params=fit_kwargs(best, y_full))
        log(f"  [{task}] 5-fold CV macro-F1 on KDDTrain+: {scores.mean():.4f} +/- {scores.std():.4f} "
            f"({time.perf_counter() - t0:.0f}s)")

        bundle = {"task": task, "model_name": best, "classes": CLASSES, "label_encoder": le,
                  "constant_cols": constant_cols, "best_params": results[best]["best_params"],
                  "cv5_f1_macro_mean": float(scores.mean()), "cv5_f1_macro_std": float(scores.std())}
        task_info = {"best_model": best, "models": results,
                     "cv5_f1_macro": {"mean": float(scores.mean()), "std": float(scores.std()),
                                      "folds": scores.tolist()}}
        if task == "binary":
            p_val = best_pipe.predict_proba(X_val)[:, 1]
            thr, rec, far, curve = choose_threshold(yval, p_val)
            log(f"  [binary] threshold={thr:.3g} -> val recall={rec:.4f}, false-alarm={far:.4f} "
                f"(constraint FAR<={MAX_FAR})")
            bundle["threshold"] = thr
            task_info["threshold"] = {"value": thr, "val_recall": rec, "val_false_alarm_rate": far,
                                      "max_far_constraint": MAX_FAR, "curve": curve}
            # thresholds for every candidate so evaluate.py can apply the same rule to all
            task_info["candidate_thresholds"] = {
                n: choose_threshold(yval, m.predict_proba(X_val)[:, 1])[0] for n, m in fitted.items()}

        # Refit best model on the full KDDTrain+ and persist the deployable bundle
        t0 = time.perf_counter()
        final = make_pipeline(best, make_estimators(task, N_CPU)[best].set_params(
            **results[best]["best_params"]), constant_cols)
        final.fit(X, y_full, **fit_kwargs(best, y_full))
        bundle["pipeline"] = final
        bundle["refit_time_s"] = time.perf_counter() - t0
        path = MODELS_DIR / f"{task}_model.joblib"
        joblib.dump(bundle, path)
        log(f"  [{task}] refit on full KDDTrain+ in {bundle['refit_time_s']:.1f}s -> {path}")
        summary["tasks"][task] = task_info

    np.save(MODELS_DIR / "val_idx.npy", idx_val)
    summary["total_train_time_s"] = time.perf_counter() - t_start
    (REPORTS_DIR / "train_summary.json").write_text(json.dumps(summary, indent=2))
    log(f"\nTraining stage done in {summary['total_train_time_s']:.0f}s")


if __name__ == "__main__":
    main()

> ⏱️ The next cell is the long one (≈ 7–8 min on 16 cores).

In [ ]:
%run src/train.py

## 5. Evaluation — `evaluate.py`
- **Protocol A:** the 15 % validation split (same distribution as training).
- **Protocol B:** the official **KDDTest+** and its hard subset **KDDTest-21** — contains 17 attack types never seen in training.

Computes accuracy, precision, recall (detection rate), macro/weighted F1, ROC-AUC, **false-alarm rate**,
training time, inference time and model size for every model; seen-vs-unseen analysis; per-class recall;
SHAP explainability; threshold sensitivity; error analysis. Writes `reports/metrics.json` and `reports/figures/`.

In [ ]:
%%writefile src/evaluate.py
"""IntruShield - evaluation under Protocol A (validation split) and Protocol B (KDDTest+/-21).

Produces reports/metrics.json and all figures in reports/figures/.
"""
import io
import json
import time
import warnings

import joblib
import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402
import shap  # noqa: E402
from sklearn.metrics import (accuracy_score, auc, confusion_matrix, f1_score,  # noqa: E402
                             precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve)

from config import (CANDIDATES_DIR, CLASSES, FEATURES, FIG_DIR, MODELS_DIR, NAVY, PALETTE,  # noqa: E402
                    RANDOM_STATE, REPORTS_DIR, TEAL)
from data import load_split  # noqa: E402

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk", font_scale=0.75)
plt.rcParams.update({"axes.edgecolor": "#cccccc", "grid.color": "#e6e6e6", "axes.titleweight": "bold",
                     "axes.titlecolor": NAVY, "figure.dpi": 110})
MODEL_ORDER = ["LogisticRegression", "RandomForest", "XGBoost", "LightGBM"]
MODEL_COLOR = dict(zip(MODEL_ORDER, PALETTE))
PROTO_COLOR = {"A: validation": PALETTE[0], "B: KDDTest+": PALETTE[1], "B: KDDTest-21": PALETTE[2]}


def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / name, bbox_inches="tight")
    plt.close(fig)
    return name


def binary_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    return {
        "accuracy": accuracy_score(y, pred), "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred), "f1_macro": f1_score(y, pred, average="macro"),
        "f1_weighted": f1_score(y, pred, average="weighted"), "roc_auc": roc_auc_score(y, p),
        "false_alarm_rate": float(pred[y == 0].mean()), "threshold": thr,
        "confusion": confusion_matrix(y, pred, labels=[0, 1]).tolist(),
    }


def multiclass_metrics(y, proba):
    pred = proba.argmax(1)
    labels = list(range(len(CLASSES)))
    present = sorted(set(y))
    try:
        roc = roc_auc_score(y, proba[:, present] / proba[:, present].sum(1, keepdims=True),
                            multi_class="ovr", average="macro", labels=present)
    except ValueError:
        roc = float("nan")
    rec_per_class = recall_score(y, pred, labels=labels, average=None, zero_division=0)
    normal = CLASSES.index("Normal")
    return {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, average="macro", zero_division=0),
        "recall": recall_score(y, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y, pred, average="macro"), "f1_weighted": f1_score(y, pred, average="weighted"),
        "roc_auc": float(roc), "false_alarm_rate": float((pred[y == normal] != normal).mean()),
        "per_class_recall": dict(zip(CLASSES, map(float, rec_per_class))),
        "per_class_f1": dict(zip(CLASSES, map(float, f1_score(y, pred, labels=labels, average=None,
                                                                 zero_division=0)))),
        "confusion": confusion_matrix(y, pred, labels=labels).tolist(),
    }


def model_size_mb(obj):
    buf = io.BytesIO()
    joblib.dump(obj, buf)
    return buf.getbuffer().nbytes / 1e6


def timed_proba(pipe, X):
    t0 = time.perf_counter()
    p = pipe.predict_proba(X)
    return p, (time.perf_counter() - t0) / len(X) * 1e6


def main():
    t_start = time.perf_counter()
    summary = json.loads((REPORTS_DIR / "train_summary.json").read_text())
    audit = json.loads((REPORTS_DIR / "audit.json").read_text())
    train, test, test21 = load_split("train"), load_split("test"), load_split("test21")
    val = train.iloc[np.load(MODELS_DIR / "val_idx.npy")]
    bundles = {t: joblib.load(MODELS_DIR / f"{t}_model.joblib") for t in ("binary", "multiclass")}
    le = bundles["multiclass"]["label_encoder"]
    best = {t: summary["tasks"][t]["best_model"] for t in bundles}
    thresholds = summary["tasks"]["binary"]["candidate_thresholds"]
    seen_attacks = set(train["attack"])
    datasets = {"A: validation": val, "B: KDDTest+": test, "B: KDDTest-21": test21}

    metrics = {"best": best, "protocols": {k: {"binary": {}, "multiclass": {}} for k in datasets},
               "model_meta": {"binary": {}, "multiclass": {}}, "deployed": {},
               "cv5": {t: summary["tasks"][t]["cv5_f1_macro"] for t in bundles},
               "threshold": {k: v for k, v in summary["tasks"]["binary"]["threshold"].items()
                             if k != "curve"},
               "split": summary["split"]}
    probas = {"binary": {}, "multiclass": {}}  # (model, dataset) -> proba on that dataset

    # ---- every candidate model, both protocols ----------------------------------------------
    for task in ("binary", "multiclass"):
        for name in MODEL_ORDER:
            pipe = joblib.load(CANDIDATES_DIR / f"{task}_{name}.joblib")
            meta = {"train_time_s": summary["tasks"][task]["models"][name]["train_time_s"],
                    "size_mb": model_size_mb(pipe),
                    "best_params": summary["tasks"][task]["models"][name]["best_params"]}
            for dname, df in datasets.items():
                X = df[FEATURES]
                p, us = timed_proba(pipe, X)
                probas[task][(name, dname)] = p
                if task == "binary":
                    m = binary_metrics(df["is_attack"].values, p[:, 1], thresholds[name])
                else:
                    m = multiclass_metrics(le.transform(df["label5"]), p)
                m["inference_us_per_row"] = us
                metrics["protocols"][dname][task][name] = m
                if dname == "B: KDDTest+":
                    meta["inference_us_per_row"] = us
            metrics["model_meta"][task][name] = meta
            print(f"[{task}] {name}: " + "  ".join(
                f"{d.split(':')[1].strip()} F1m={metrics['protocols'][d][task][name]['f1_macro']:.4f}"
                for d in datasets), flush=True)

    # ---- deployed models (refit on 100% of KDDTrain+) on Protocol B -------------------------
    for task, b in bundles.items():
        metrics["deployed"][task] = {"model": b["model_name"], "refit_time_s": b["refit_time_s"],
                                     "size_mb": model_size_mb(b["pipeline"])}
        for dname in ("B: KDDTest+", "B: KDDTest-21"):
            df = datasets[dname]
            p, us = timed_proba(b["pipeline"], df[FEATURES])
            probas[task][("deployed", dname)] = p
            m = (binary_metrics(df["is_attack"].values, p[:, 1], b["threshold"]) if task == "binary"
                 else multiclass_metrics(le.transform(df["label5"]), p))
            m["inference_us_per_row"] = us
            metrics["deployed"][task][dname] = m

    # ---- leakage check -------------------------------------------------------------------------
    max_acc_a = max(m["accuracy"] for t in ("binary", "multiclass")
                    for m in metrics["protocols"]["A: validation"][t].values())
    metrics["leakage_check"] = {
        "max_protocol_a_accuracy": max_acc_a, "triggered": bool(max_acc_a > 0.999),
        "val_rows_with_identical_feature_vector_in_train_split": summary["split"]["val_rows_duplicated_in_train"],
        "val_rows": summary["split"]["val_rows"],
        "exact_duplicate_rows_in_kddtrain": audit["duplicate_rows"]["train"],
        "note": ("Protocol A accuracy > 99.9%: checked duplicates between train and validation "
                 "splits; see counts. NSL-KDD was de-duplicated by its authors, so near-perfect "
                 "same-distribution accuracy reflects an easy in-distribution task, not leakage."
                 if max_acc_a > 0.999 else "Protocol A accuracy <= 99.9%; no leakage check needed."),
    }

    # ---- seen vs unseen attacks on KDDTest+ (binary detection rate) --------------------------
    test_attack = test[test["is_attack"] == 1].copy()
    test_attack["seen"] = test_attack["attack"].isin(seen_attacks)
    svu = {}
    for name in MODEL_ORDER + ["deployed"]:
        p = probas["binary"][(name, "B: KDDTest+")][:, 1][test["is_attack"].values == 1]
        thr = bundles["binary"]["threshold"] if name == "deployed" else thresholds[name]
        det = p >= thr
        svu[name] = {
            grp: {"n": int(mask.sum()), "detected": int(det[mask].sum()),
                  "recall": float(det[mask].mean())}
            for grp, mask in (("seen", test_attack["seen"].values), ("unseen", ~test_attack["seen"].values))}
    metrics["seen_vs_unseen"] = svu

    # ---- error analysis (deployed models on KDDTest+) ----------------------------------------
    p_bin = probas["binary"][("deployed", "B: KDDTest+")][:, 1]
    det_all = p_bin >= bundles["binary"]["threshold"]
    pred_mc = le.inverse_transform(probas["multiclass"][("deployed", "B: KDDTest+")].argmax(1))
    rows = []
    for atk, g in test[test["is_attack"] == 1].groupby("attack"):
        idx = g.index.values
        missed = int((~det_all[idx]).sum())
        mc_wrong = int((pred_mc[idx] != g["label5"].values).sum())
        top_conf = pd.Series(pred_mc[idx]).value_counts().index[0]
        rows.append({"attack": atk, "class": g["label5"].iloc[0], "n": int(len(g)),
                     "seen_in_train": atk in seen_attacks,
                     "binary_missed": missed, "binary_miss_rate": missed / len(g),
                     "multiclass_wrong": mc_wrong, "multiclass_error_rate": mc_wrong / len(g),
                     "multiclass_most_common_prediction": str(top_conf),
                     "mean_difficulty": float(g["difficulty"].mean())})
    per_attack = pd.DataFrame(rows).sort_values("binary_missed", ascending=False)
    metrics["error_analysis"] = {
        "binary_total_missed_attacks": int((~det_all[test["is_attack"].values == 1]).sum()),
        "binary_false_alarms": int(det_all[test["is_attack"].values == 0].sum()),
        "per_attack_type": per_attack.to_dict(orient="records"),
        "most_missed_binary": per_attack.head(10)[["attack", "class", "n", "binary_missed",
                                                   "binary_miss_rate", "seen_in_train"]].to_dict(orient="records"),
        "most_misclassified_multiclass": per_attack.sort_values("multiclass_wrong", ascending=False)
        .head(10)[["attack", "class", "n", "multiclass_wrong", "multiclass_error_rate",
                   "multiclass_most_common_prediction", "seen_in_train"]].to_dict(orient="records"),
    }

    # ======================= FIGURES =========================================================
    figs = {}
    # 1. model comparison (macro-F1, both protocols)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
    for ax, task in zip(axes, ("binary", "multiclass")):
        rec = [{"model": n, "protocol": d, "f1": metrics["protocols"][d][task][n]["f1_macro"]}
               for d in datasets for n in MODEL_ORDER]
        sns.barplot(pd.DataFrame(rec), x="model", y="f1", hue="protocol", palette=PROTO_COLOR, ax=ax,
                    order=MODEL_ORDER, width=0.75, edgecolor="white", linewidth=1.5)
        for c in ax.containers:
            ax.bar_label(c, fmt="%.3f", fontsize=8, padding=1)
        ax.set(title=f"{task.capitalize()} - macro-F1", xlabel="", ylabel="macro-F1", ylim=(0, 1.08))
        ax.tick_params(axis="x", rotation=15)
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, fontsize=8, title="", frameon=False)
    figs["model_comparison"] = save(fig, "model_comparison.png")

    # 2. confusion matrices (deployed models, KDDTest+)
    for task, labels in (("binary", ["normal", "attack"]), ("multiclass", CLASSES)):
        cm = np.array(metrics["deployed"][task]["B: KDDTest+"]["confusion"])
        fig, axes = plt.subplots(1, 2, figsize=(11.5 if task == "multiclass" else 9, 4.6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
                    xticklabels=labels, yticklabels=labels)
        axes[0].set(title=f"{task.capitalize()} ({best[task]}) - counts", xlabel="Predicted", ylabel="True")
        sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                    cbar=False, ax=axes[1], xticklabels=labels, yticklabels=labels)
        axes[1].set(title="Row-normalised (recall per class)", xlabel="Predicted", ylabel="")
        figs[f"confusion_{task}"] = save(fig, f"confusion_{task}.png")

    # 3. ROC + PR curves (binary, all models) on validation and KDDTest+
    fig, axes = plt.subplots(2, 2, figsize=(12, 9.5))
    for row, dname in enumerate(("A: validation", "B: KDDTest+")):
        y = datasets[dname]["is_attack"].values
        for name in MODEL_ORDER:
            p = probas["binary"][(name, dname)][:, 1]
            fpr, tpr, _ = roc_curve(y, p)
            prec, rec, _ = precision_recall_curve(y, p)
            axes[row, 0].plot(fpr, tpr, color=MODEL_COLOR[name], lw=2,
                              label=f"{name} (AUC {auc(fpr, tpr):.4f})")
            axes[row, 1].plot(rec, prec, color=MODEL_COLOR[name], lw=2,
                              label=f"{name} (AP {auc(rec, prec):.4f})")
        axes[row, 0].plot([0, 1], [0, 1], "--", color="#999", lw=1)
        axes[row, 0].set(title=f"ROC - {dname}", xlabel="False alarm rate (FPR)", ylabel="Detection rate (TPR)")
        axes[row, 1].set(title=f"Precision-Recall - {dname}", xlabel="Recall", ylabel="Precision")
        for ax in axes[row]:
            ax.legend(fontsize=8, loc="lower right" if ax is axes[row, 0] else "lower left")
    axes[0, 0].set_xlim(-0.005, 0.25)  # zoom into the useful low-FPR corner for validation
    figs["roc_pr_curves"] = save(fig, "roc_pr_curves.png")

    # 4. per-class recall (multiclass best model), Protocol A vs B, rare classes highlighted
    rec = [{"class": c, "protocol": d, "recall": metrics["protocols"][d]["multiclass"][best["multiclass"]]["per_class_recall"][c]}
           for d in datasets for c in CLASSES]
    fig, ax = plt.subplots(figsize=(10, 4.8))
    sns.barplot(pd.DataFrame(rec), x="class", y="recall", hue="protocol", palette=PROTO_COLOR, ax=ax,
                order=CLASSES, edgecolor="white", linewidth=1.5)
    for c in ax.containers:
        ax.bar_label(c, fmt="%.2f", fontsize=8, padding=1)
    for i, c in enumerate(CLASSES):
        if c in ("R2L", "U2R"):
            ax.axvspan(i - 0.5, i + 0.5, color=PALETTE[2], alpha=0.10, zorder=0)
            ax.text(i, 1.03, "rare class", ha="center", fontsize=9, color=PALETTE[2], fontweight="bold")
    ax.set(title=f"Per-class recall - {best['multiclass']}", ylim=(0, 1.1), xlabel="", ylabel="Recall")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8, title="", frameon=False)
    figs["per_class_recall"] = save(fig, "per_class_recall.png")

    # 5. seen vs unseen attacks (binary detection rate on KDDTest+)
    rec = [{"model": n, "group": f"{g} in training\n(n={svu[n][g]['n']})", "recall": svu[n][g]["recall"]}
           for n in MODEL_ORDER for g in ("seen", "unseen")]
    fig, ax = plt.subplots(figsize=(10, 4.8))
    sns.barplot(pd.DataFrame(rec), x="model", y="recall", hue="group", palette=[PALETTE[1], PALETTE[4]],
                ax=ax, order=MODEL_ORDER, edgecolor="white", linewidth=1.5)
    for c in ax.containers:
        ax.bar_label(c, fmt="%.3f", fontsize=8, padding=1)
    ax.set(title="KDDTest+ detection rate: attack types SEEN vs UNSEEN in KDDTrain+",
           ylim=(0, 1.1), xlabel="", ylabel="Attack recall (detection rate)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=8, title="", frameon=False)
    figs["seen_vs_unseen"] = save(fig, "seen_vs_unseen.png")

    # 5b. per attack type detection (unseen types) for the deployed binary model
    pa = per_attack[per_attack["n"] >= 5].sort_values(["seen_in_train", "binary_miss_rate"],
                                                       ascending=[True, False])
    fig, ax = plt.subplots(figsize=(11, 6.5))
    colors = [PALETTE[4] if not s else PALETTE[1] for s in pa["seen_in_train"]]
    ax.barh(pa["attack"], 1 - pa["binary_miss_rate"], color=colors, edgecolor="white")
    for i, (r, n) in enumerate(zip(1 - pa["binary_miss_rate"], pa["n"])):
        ax.text(r + 0.01, i, f"{r:.2f}  (n={n})", va="center", fontsize=7.5)
    ax.invert_yaxis()
    ax.set(title=f"Detection rate per attack type on KDDTest+ ({best['binary']}, n>=5)",
           xlabel="Detection rate", xlim=(0, 1.25))
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=PALETTE[4]), plt.Rectangle((0, 0), 1, 1, color=PALETTE[1])],
              labels=["unseen in training", "seen in training"], fontsize=8, loc="upper center",
              bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)
    figs["per_attack_detection"] = save(fig, "per_attack_detection.png")

    # 6. SHAP summary + top-15 importance for the best tree model (binary task)
    tree_models = [n for n in ("LightGBM", "XGBoost", "RandomForest")]
    shap_model = best["binary"] if best["binary"] in tree_models else max(
        tree_models, key=lambda n: metrics["protocols"]["A: validation"]["binary"][n]["f1_macro"])
    pipe = bundles["binary"]["pipeline"] if shap_model == best["binary"] else joblib.load(
        CANDIDATES_DIR / f"binary_{shap_model}.joblib")
    n_shap = 2000 if shap_model != "RandomForest" else 500  # RF TreeExplainer is ~50x slower
    sample = val.sample(n=n_shap, random_state=RANDOM_STATE)
    Xt = pipe["prep"].transform(sample[FEATURES])
    feat_names = list(pipe["prep"].get_feature_names_out())
    t0 = time.perf_counter()
    explainer = shap.TreeExplainer(pipe["clf"])
    sv = explainer.shap_values(Xt)
    if isinstance(sv, list):
        sv = sv[1]
    elif sv.ndim == 3:
        sv = sv[:, :, 1]
    shap_time = time.perf_counter() - t0
    imp = pd.Series(np.abs(sv).mean(0), index=feat_names).sort_values(ascending=False)
    plt.figure(figsize=(9, 8))
    shap.summary_plot(sv, Xt, feature_names=feat_names, max_display=20, show=False, plot_size=None)
    plt.title(f"SHAP summary - {shap_model} (binary, {n_shap} validation rows)", color=NAVY, fontweight="bold")
    figs["shap_summary"] = save(plt.gcf(), "shap_summary.png")
    fig, ax = plt.subplots(figsize=(9, 6))
    top = imp.head(15)[::-1]
    ax.barh(top.index, top.values, color=TEAL, edgecolor="white")
    ax.set(title=f"Top 15 features by mean |SHAP| - {shap_model}", xlabel="mean |SHAP value| (log-odds)")
    figs["feature_importance"] = save(fig, "feature_importance.png")
    metrics["shap"] = {"model": shap_model, "n_rows": n_shap, "time_s": shap_time,
                       "top15": [{"feature": f, "mean_abs_shap": float(v)} for f, v in imp.head(15).items()]}

    # 7. threshold vs recall / false-alarm (validation) + sensitivity table (val and KDDTest+)
    curve = pd.DataFrame(summary["tasks"]["binary"]["threshold"]["curve"])
    thr_info = summary["tasks"]["binary"]["threshold"]
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.plot(curve["threshold"], curve["recall"], color=PALETTE[1], lw=2.2, label="Attack recall")
    ax.plot(curve["threshold"], curve["false_alarm_rate"], color=PALETTE[4], lw=2.2, label="False-alarm rate")
    ax.axhline(thr_info["max_far_constraint"], ls="--", color="#888", lw=1, label="FAR budget (5%)")
    ax.axvline(thr_info["value"], ls=":", color=NAVY, lw=1.5,
               label=f"chosen threshold = {thr_info['value']:.3g}")
    ax.set_xscale("log")
    ax.set(title=f"Decision threshold trade-off on validation ({best['binary']})",
           xlabel="Threshold on P(attack)  (log scale)", ylabel="Rate", ylim=(0, 1.05))
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4, fontsize=8, frameon=False)
    figs["threshold_curve"] = save(fig, "threshold_curve.png")
    sens = []
    yv, yt = val["is_attack"].values, test["is_attack"].values
    pv = probas["binary"][(best["binary"], "A: validation")][:, 1]
    pt = probas["binary"][(best["binary"], "B: KDDTest+")][:, 1]
    for label, cut in [("chosen (max recall, FAR<=5%)", thr_info["value"]), ("0.01", 0.01), ("0.1", 0.1),
                       ("0.5 (default)", 0.5), ("0.9", 0.9)]:
        sens.append({"label": label, "threshold": cut,
                     "val_recall": float((pv[yv == 1] >= cut).mean()), "val_far": float((pv[yv == 0] >= cut).mean()),
                     "test_recall": float((pt[yt == 1] >= cut).mean()), "test_far": float((pt[yt == 0] >= cut).mean()),
                     "test_f1_macro": float(f1_score(yt, (pt >= cut).astype(int), average="macro"))})
    metrics["threshold_sensitivity"] = sens

    metrics["figures"] = figs
    metrics["eval_time_s"] = time.perf_counter() - t_start
    (REPORTS_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, default=float))
    print(f"\nEvaluation done in {metrics['eval_time_s']:.0f}s -> {REPORTS_DIR / 'metrics.json'}")


if __name__ == "__main__":
    main()

In [ ]:
%run src/evaluate.py

## 6. Results

In [ ]:
M = json.loads((ROOT / "reports" / "metrics.json").read_text())
P = M["protocols"]
ORDER = ["LogisticRegression", "RandomForest", "XGBoost", "LightGBM"]

def table(task):
    rows = []
    for name in ORDER:
        a, b, c = P["A: validation"][task][name], P["B: KDDTest+"][task][name], P["B: KDDTest-21"][task][name]
        rows.append({"model": name + (" ★" if name == M["best"][task] else ""),
                     "F1m A": a["f1_macro"], "F1m B": b["f1_macro"], "F1m B-21": c["f1_macro"],
                     "Acc A": a["accuracy"], "Acc B": b["accuracy"],
                     "Recall B": b["recall"], "FAR B": b["false_alarm_rate"], "AUC B": b["roc_auc"]})
    return pd.DataFrame(rows).set_index("model").round(4)

print("BINARY (normal vs attack)   ★ = selected on validation macro-F1")
display(table("binary"))
print("5-CLASS (Normal / DoS / Probe / R2L / U2R)")
display(table("multiclass"))
print("5-fold CV on KDDTrain+:", {k: f"{v['mean']:.4f} ± {v['std']:.4f}" for k, v in M["cv5"].items()})

In [ ]:
db = M["deployed"]["binary"]["B: KDDTest+"]
svu = M["seen_vs_unseen"]["deployed"]
print(f"Deployed binary {M['best']['binary']}  (threshold {M['threshold']['value']:.3g}) on KDDTest+")
print(f"  accuracy {db['accuracy']:.2%} | recall {db['recall']:.2%} | false alarms {db['false_alarm_rate']:.2%} | ROC-AUC {db['roc_auc']:.4f}")
print(f"  attacks of SEEN types detected:   {svu['seen']['recall']:.1%}  ({svu['seen']['detected']:,} / {svu['seen']['n']:,})")
print(f"  attacks of UNSEEN types detected: {svu['unseen']['recall']:.1%}  ({svu['unseen']['detected']:,} / {svu['unseen']['n']:,})")

pc = P["B: KDDTest+"]["multiclass"][M["best"]["multiclass"]]
display(pd.DataFrame({"recall (KDDTest+)": pc["per_class_recall"], "F1 (KDDTest+)": pc["per_class_f1"]}).round(4))

print("Threshold sensitivity (analysis only — the threshold was fixed on validation before KDDTest+ was read)")
display(pd.DataFrame(M["threshold_sensitivity"]).set_index("label").round(4))

print("Most-missed attack types (binary)")
display(pd.DataFrame(M["error_analysis"]["most_missed_binary"]).set_index("attack"))

### Figures

In [ ]:
from IPython.display import Image, Markdown, display

CAPTIONS = {
    "model_comparison": "Macro-F1 of every model: Protocol A vs KDDTest+ vs KDDTest-21",
    "roc_pr_curves": "ROC and Precision-Recall curves (binary)",
    "seen_vs_unseen": "Detection rate on attack types seen vs never seen in training",
    "per_attack_detection": "Detection rate per attack type on KDDTest+",
    "per_class_recall": "Per-class recall under both protocols",
    "confusion_multiclass": "5-class confusion matrices on KDDTest+",
    "confusion_binary": "Binary confusion matrices on KDDTest+",
    "shap_summary": "SHAP beeswarm — which features push a connection toward 'attack'",
    "feature_importance": "Top features by mean |SHAP|",
    "threshold_curve": "Recall vs false-alarm rate across decision thresholds (validation)",
}
for key, fname in M["figures"].items():
    display(Markdown(f"**{CAPTIONS.get(key, key)}**"))
    display(Image(filename=str(ROOT / "reports" / "figures" / fname), width=900))

## 7. HTML report — `report.py`
Builds one self-contained `reports/eval_report.html` (all figures embedded) with 11 sections:
Problem · Dataset & Audit · Preprocessing · Models Compared · Protocol A vs B · Unseen attacks ·
Per-class · SHAP · Threshold & False Alarms · Error Analysis · Limitations.

In [ ]:
%%writefile src/report.py
"""IntruShield - build the self-contained HTML evaluation report from metrics.json + audit.json."""
import base64
import json
from datetime import datetime

from config import CLASSES, FIG_DIR, REPORTS_DIR

MODEL_ORDER = ["LogisticRegression", "RandomForest", "XGBoost", "LightGBM"]
CSS = """
:root{--navy:#0b2545;--teal:#13a4a4;--ink:#1f2933;--muted:#5f6b7a;--bg:#f6f8fa;--card:#fff;--line:#e3e8ee}
*{box-sizing:border-box}body{margin:0;font-family:Segoe UI,Inter,Arial,sans-serif;color:var(--ink);background:var(--bg);line-height:1.55}
header{background:linear-gradient(120deg,var(--navy),#123a6b 70%,var(--teal));color:#fff;padding:40px 48px}
header h1{margin:0;font-size:34px;letter-spacing:.5px}header p{margin:8px 0 0;opacity:.9;font-size:16px}
nav{position:sticky;top:0;background:var(--navy);padding:8px 48px;z-index:9;overflow-x:auto;white-space:nowrap}
nav a{color:#cfe8e8;text-decoration:none;margin-right:18px;font-size:13px}nav a:hover{color:#fff}
main{max-width:1180px;margin:0 auto;padding:24px 32px 60px}
section{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:26px 30px;margin:22px 0;box-shadow:0 1px 3px rgba(11,37,69,.06)}
h2{margin:0 0 14px;color:var(--navy);font-size:22px;border-left:5px solid var(--teal);padding-left:12px}
h3{color:var(--navy);margin:20px 0 8px;font-size:16px}
table{border-collapse:collapse;width:100%;font-size:13.5px;margin:10px 0}
th{background:var(--navy);color:#fff;text-align:left;padding:8px 10px;font-weight:600}
td{padding:7px 10px;border-bottom:1px solid var(--line)}tr:nth-child(even) td{background:#f9fbfc}
td.num,th.num{text-align:right;font-variant-numeric:tabular-nums}
.best td{font-weight:700;background:#e8f6f6!important}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:14px;margin:12px 0 4px}
.kpi{background:linear-gradient(160deg,#f2fafa,#fff);border:1px solid #cfe8e8;border-radius:10px;padding:14px 16px}
.kpi .v{font-size:26px;font-weight:700;color:var(--navy)}.kpi .l{font-size:12px;color:var(--muted);text-transform:uppercase;letter-spacing:.5px}
figure{margin:18px 0;text-align:center}figure img{max-width:100%;border:1px solid var(--line);border-radius:6px}
figcaption{font-size:12.5px;color:var(--muted);margin-top:6px}
.note{background:#fff8e6;border-left:4px solid #e0a800;padding:10px 14px;border-radius:6px;font-size:14px}
.ok{background:#eaf7f1;border-left:4px solid #2e9e6b;padding:10px 14px;border-radius:6px;font-size:14px}
.cols{display:grid;grid-template-columns:1fr 1fr;gap:20px}@media(max-width:800px){.cols{grid-template-columns:1fr}}
code{background:#eef2f6;padding:1px 5px;border-radius:4px;font-size:12.5px}
footer{text-align:center;color:var(--muted);font-size:12px;padding:20px}
"""


def img(name, caption):
    data = base64.b64encode((FIG_DIR / name).read_bytes()).decode()
    return f'<figure><img src="data:image/png;base64,{data}" alt="{caption}"><figcaption>{caption}</figcaption></figure>'


def pct(x, d=2):
    return f"{100 * x:.{d}f}%"


def f4(x):
    return f"{x:.4f}"


def table(headers, rows, best_row=None, num_from=1):
    h = "".join(f'<th{" class=num" if i >= num_from else ""}>{x}</th>' for i, x in enumerate(headers))
    body = ""
    for r_i, r in enumerate(rows):
        cls = ' class="best"' if best_row is not None and r_i == best_row else ""
        body += f"<tr{cls}>" + "".join(
            f'<td{" class=num" if i >= num_from else ""}>{x}</td>' for i, x in enumerate(r)) + "</tr>"
    return f"<table><thead><tr>{h}</tr></thead><tbody>{body}</tbody></table>"


def main():
    M = json.loads((REPORTS_DIR / "metrics.json").read_text())
    A = json.loads((REPORTS_DIR / "audit.json").read_text())
    P = M["protocols"]
    best_b, best_m = M["best"]["binary"], M["best"]["multiclass"]
    dep_b, dep_m = M["deployed"]["binary"]["B: KDDTest+"], M["deployed"]["multiclass"]["B: KDDTest+"]
    dep_b21 = M["deployed"]["binary"]["B: KDDTest-21"]
    thr = M["threshold"]
    svu = M["seen_vs_unseen"]
    unseen_rows = A["unseen_attack_rows_in_test"]
    unseen_types = list(A["attacks_in_test_not_in_train"].keys())
    total_test_attacks = A["binary_distribution"]["test"]["attack"]

    def model_table(task):
        rows, best_i = [], None
        for i, n in enumerate(MODEL_ORDER):
            a, b, b21 = P["A: validation"][task][n], P["B: KDDTest+"][task][n], P["B: KDDTest-21"][task][n]
            meta = M["model_meta"][task][n]
            if n == M["best"][task]:
                best_i = i
            rows.append([n, f4(a["f1_macro"]), f4(b["f1_macro"]), f4(b21["f1_macro"]),
                         pct(a["accuracy"]), pct(b["accuracy"]), pct(a["recall"]), pct(b["recall"]),
                         pct(a["false_alarm_rate"]), pct(b["false_alarm_rate"]),
                         f4(a["roc_auc"]), f4(b["roc_auc"]),
                         f"{meta['train_time_s']:.1f}", f"{meta['inference_us_per_row']:.1f}", f"{meta['size_mb']:.2f}"])
        return table(["Model", "F1m A", "F1m B", "F1m B-21", "Acc A", "Acc B", "Recall A", "Recall B",
                      "FAR A", "FAR B", "AUC A", "AUC B", "Train s", "Infer µs/row", "Size MB"], rows, best_i)

    def full_metric_table(task, dname):
        rows = []
        for n in MODEL_ORDER:
            m = P[dname][task][n]
            rows.append([n, pct(m["accuracy"]), f4(m["precision"]), f4(m["recall"]), f4(m["f1_macro"]),
                         f4(m["f1_weighted"]), f4(m["roc_auc"]), pct(m["false_alarm_rate"])])
        return table(["Model", "Accuracy", "Precision", "Recall (DR)", "F1 macro", "F1 weighted", "ROC-AUC",
                      "False alarm"], rows, MODEL_ORDER.index(M["best"][task]))

    # ---- sections --------------------------------------------------------------------------
    kpis = f"""
<div class="kpis">
<div class="kpi"><div class="v">{pct(dep_b['recall'], 1)}</div><div class="l">Attack detection rate · KDDTest+</div></div>
<div class="kpi"><div class="v">{pct(dep_b['false_alarm_rate'], 1)}</div><div class="l">False-alarm rate · KDDTest+</div></div>
<div class="kpi"><div class="v">{f4(dep_b['roc_auc'])}</div><div class="l">ROC-AUC (binary) · KDDTest+</div></div>
<div class="kpi"><div class="v">{f4(dep_m['f1_macro'])}</div><div class="l">Macro-F1 (5-class) · KDDTest+</div></div>
<div class="kpi"><div class="v">{f4(P['A: validation']['binary'][best_b]['f1_macro'])}</div><div class="l">Macro-F1 (binary) · validation</div></div>
<div class="kpi"><div class="v">{M['model_meta']['binary'][best_b]['inference_us_per_row']:.0f} µs</div><div class="l">Inference per connection</div></div>
</div>"""

    dist = lambda d: ", ".join(f"{k}: {v:,}" for k, v in d.items())  # noqa: E731
    unseen_tbl = table(["Attack type (test only)", "Class", "Rows in KDDTest+"],
                       [[k, v["class"], f"{v['test_rows']:,}"] for k, v in
                        sorted(A["attacks_in_test_not_in_train"].items(), key=lambda kv: -kv[1]["test_rows"])], num_from=2)

    audit_sec = f"""
<p>NSL-KDD is the de-duplicated revision of KDD'99. Each row is one TCP/UDP/ICMP connection described by
41 features (9 basic, 13 content, 9 time-based traffic, 10 host-based traffic) plus an attack label and a
difficulty score (dropped from the feature set, kept for analysis).</p>
{table(["", "KDDTrain+", "KDDTest+", "KDDTest-21"], [
    ["Rows", f"{A['rows']['train']:,}", f"{A['rows']['test']:,}", f"{A['rows']['test21']:,}"],
    ["Exact duplicate rows", str(A['duplicate_rows']['train']), str(A['duplicate_rows']['test']), "-"],
    ["Test rows whose feature vector appears in train", "-", str(A['duplicate_rows']['test_rows_also_in_train']), "-"],
    ["Distinct attack types", str(A['n_attack_types']['train']), str(A['n_attack_types']['test']), "-"],
    ["5-class distribution", dist(A['class_distribution']['train']), dist(A['class_distribution']['test']), dist(A['class_distribution']['test21'])],
    ["Binary distribution", dist(A['binary_distribution']['train']), dist(A['binary_distribution']['test']), "-"],
    ["Constant columns (dropped)", ", ".join(A['constant_columns']) or "none", "", ""],
])}
<h3>Attack types present in KDDTest+ but absent from KDDTrain+</h3>
<p>{len(unseen_types)} attack types, <b>{unseen_rows:,} rows ({pct(unseen_rows / total_test_attacks, 1)} of all test attacks)</b>,
never appear in training. This is the designed difficulty of NSL-KDD and the main driver of the Protocol A → B gap.</p>
{unseen_tbl}"""

    prep_sec = f"""
<p>A single scikit-learn <code>Pipeline</code> (<code>ColumnTransformer</code> → classifier) is fitted on training
rows only, and persisted together with the model, so test rows always pass through exactly the training-time
transformation.</p>
<ul>
<li><b>Categorical</b> (<code>protocol_type</code>, <code>service</code>, <code>flag</code>): <code>OneHotEncoder(handle_unknown="ignore")</code> – services seen only in test become all-zero rather than crashing.</li>
<li><b>Heavily skewed counts/bytes</b> (duration, src_bytes, dst_bytes, hot, count, srv_count, dst_host_count, …): <code>log1p</code>.</li>
<li><b>Scaling</b>: <code>StandardScaler</code> on all numeric columns for LogisticRegression only; trees use raw (log-transformed) values.</li>
<li><b>Dropped</b>: constant column(s) {', '.join(A['constant_columns'])}; the difficulty score.</li>
<li><b>Imbalance</b>: <code>class_weight="balanced"</code> (LR, LightGBM), <code>balanced_subsample</code> (RandomForest), balanced <code>sample_weight</code> (XGBoost).</li>
</ul>
<p><b>Leakage discipline:</b> KDDTrain+ was split 85/15 stratified on the 5-class label
({M['split']['train_rows']:,} / {M['split']['val_rows']:,} rows, <code>random_state=42</code>). Hyper-parameter search
(RandomizedSearchCV, 12 configurations × 3-fold, macro-F1) ran on the 85 % part, model selection and the decision
threshold used only the 15 % validation part. KDDTest+ / KDDTest-21 were loaded for the first time in
<code>evaluate.py</code>.</p>"""

    def params(task, n):
        p = M["model_meta"][task][n]["best_params"]
        return ", ".join(f"{k}={v:.3g}" if isinstance(v, float) else f"{k}={v}" for k, v in p.items()) or "defaults"

    models_sec = f"""
{table(["Model", "Role", "Imbalance handling", "Tuned params (binary)", "Tuned params (multiclass)"], [
    ["LogisticRegression", "linear baseline", "class_weight=balanced", "C=1 (not tuned)", "C=1 (not tuned)"],
    ["RandomForest", "bagging, 200 trees", "class_weight=balanced_subsample", "not tuned", "not tuned"],
    ["XGBoost", "gradient boosting (hist)", "balanced sample_weight", params("binary", "XGBoost"), params("multiclass", "XGBoost")],
    ["LightGBM", "gradient boosting (leaf-wise)", "class_weight=balanced", params("binary", "LightGBM"), params("multiclass", "LightGBM")],
], num_from=99)}
<p>Selected by validation macro-F1: <b>{best_b}</b> for the binary task and <b>{best_m}</b> for the 5-class task.
5-fold stratified CV of the selected configuration on the whole of KDDTrain+:
binary macro-F1 <b>{M['cv5']['binary']['mean']:.4f} ± {M['cv5']['binary']['std']:.4f}</b>,
multiclass macro-F1 <b>{M['cv5']['multiclass']['mean']:.4f} ± {M['cv5']['multiclass']['std']:.4f}</b>.</p>"""

    leak = M["leakage_check"]
    leak_html = (f'<div class="note"><b>Leakage check triggered</b> (max Protocol A accuracy {pct(leak["max_protocol_a_accuracy"])} &gt; 99.9 %). '
                 f'{leak["val_rows_with_identical_feature_vector_in_train_split"]} of {leak["val_rows"]:,} validation rows have a feature vector '
                 f'identical to a training-split row; KDDTrain+ contains {leak["exact_duplicate_rows_in_kddtrain"]} exact duplicate rows. '
                 f'The near-perfect in-distribution score therefore is not explained by duplicate leakage.</div>'
                 if leak["triggered"] else f'<div class="ok">Max Protocol A accuracy {pct(leak["max_protocol_a_accuracy"])} ≤ 99.9 %; leakage check not triggered.</div>')

    results_sec = f"""
<p><b>Protocol A</b> = the 15 % stratified hold-out of KDDTrain+ (same distribution as training).
<b>Protocol B</b> = the official KDDTest+ (and its hard subset KDDTest-21, which removes every record that 21 KDD'99
learners all classified correctly). Binary metrics use each model's own validation-chosen threshold; FAR = share of
normal connections flagged as attacks.</p>
<h3>Binary (normal vs attack)</h3>{model_table("binary")}
<h3>Multiclass (Normal / DoS / Probe / R2L / U2R)</h3>{model_table("multiclass")}
{leak_html}
{img(M['figures']['model_comparison'], "Macro-F1 of every model under Protocol A (validation), Protocol B (KDDTest+) and KDDTest-21.")}
<h3>Full metric set – Protocol B, KDDTest+</h3>
<div class="cols"><div><h3>Binary</h3>{full_metric_table("binary", "B: KDDTest+")}</div>
<div><h3>Multiclass</h3>{full_metric_table("multiclass", "B: KDDTest+")}</div></div>
<h3>Deployed models (refit on 100 % of KDDTrain+)</h3>
{table(["Model", "Dataset", "Accuracy", "Precision", "Recall", "F1 macro", "ROC-AUC", "False alarm"], [
    [f"{best_b} (binary)", "KDDTest+", pct(dep_b['accuracy']), f4(dep_b['precision']), f4(dep_b['recall']), f4(dep_b['f1_macro']), f4(dep_b['roc_auc']), pct(dep_b['false_alarm_rate'])],
    [f"{best_b} (binary)", "KDDTest-21", pct(dep_b21['accuracy']), f4(dep_b21['precision']), f4(dep_b21['recall']), f4(dep_b21['f1_macro']), f4(dep_b21['roc_auc']), pct(dep_b21['false_alarm_rate'])],
    [f"{best_m} (5-class)", "KDDTest+", pct(dep_m['accuracy']), f4(dep_m['precision']), f4(dep_m['recall']), f4(dep_m['f1_macro']), f4(dep_m['roc_auc']), pct(dep_m['false_alarm_rate'])],
], num_from=2)}
{img(M['figures']['roc_pr_curves'], "ROC (left, x-axis zoomed for validation) and Precision-Recall (right) curves of all binary models on validation (top) and KDDTest+ (bottom).")}"""

    gap_a = P["A: validation"]["binary"][best_b]["f1_macro"]
    gap_b = P["B: KDDTest+"]["binary"][best_b]["f1_macro"]
    s, u = svu[best_b]["seen"], svu[best_b]["unseen"]
    gap_sec = f"""
<p>For {best_b} the binary macro-F1 drops from <b>{f4(gap_a)}</b> (Protocol A) to <b>{f4(gap_b)}</b> (KDDTest+).
Splitting KDDTest+ attacks by whether their attack type ever occurred in KDDTrain+ explains the drop:</p>
{table(["Model", "Seen types: n", "Seen: detected", "Seen: recall", "Unseen types: n", "Unseen: detected", "Unseen: recall"], [
    [n, f"{svu[n]['seen']['n']:,}", f"{svu[n]['seen']['detected']:,}", pct(svu[n]['seen']['recall'], 1),
     f"{svu[n]['unseen']['n']:,}", f"{svu[n]['unseen']['detected']:,}", pct(svu[n]['unseen']['recall'], 1)] for n in MODEL_ORDER], MODEL_ORDER.index(best_b))}
<p>{best_b} detects <b>{pct(s['recall'], 1)}</b> of attacks whose type it has seen but only <b>{pct(u['recall'], 1)}</b> of the
{u['n']:,} connections belonging to the {len(unseen_types)} unseen types. In addition, several <i>seen</i> R2L types
(e.g. guess_passwd, warezmaster) appear in test with far higher frequency and different feature profiles than in
training, and many R2L/U2R connections are content-level attacks that look like normal logins at the connection-feature
level. Nothing in this gap is a bug: it is the covariate and label shift NSL-KDD was built to expose, and it is the
honest number to quote for deployment.</p>
{img(M['figures']['seen_vs_unseen'], "Detection rate on KDDTest+ attacks, split by whether the attack type occurs in KDDTrain+ (binary task, validation-chosen thresholds).")}
{img(M['figures']['per_attack_detection'], "Detection rate per attack type on KDDTest+ (types with at least 5 rows).")}"""

    pcr_rows = []
    for c in CLASSES:
        pcr_rows.append([c, f"{A['class_distribution']['train'].get(c, 0):,}", f"{A['class_distribution']['test'].get(c, 0):,}",
                         f4(P["A: validation"]["multiclass"][best_m]["per_class_recall"][c]),
                         f4(P["B: KDDTest+"]["multiclass"][best_m]["per_class_recall"][c]),
                         f4(P["B: KDDTest+"]["multiclass"][best_m]["per_class_f1"][c])])
    perclass_sec = f"""
{table(["Class", "Train rows", "Test rows", "Recall A (val)", "Recall B (KDDTest+)", "F1 B (KDDTest+)"], pcr_rows)}
<p>DoS and Probe transfer well. R2L and U2R are rare in training ({A['class_distribution']['train'].get('R2L', 0):,} and
{A['class_distribution']['train'].get('U2R', 0):,} rows) but much more frequent in test
({A['class_distribution']['test'].get('R2L', 0):,} and {A['class_distribution']['test'].get('U2R', 0):,}), and most of their
test rows are of unseen types – they are the weak spot of every model.</p>
{img(M['figures']['per_class_recall'], f"Per-class recall of {best_m} under both protocols; the rare R2L/U2R classes are highlighted.")}
{img(M['figures']['confusion_multiclass'], f"Confusion matrices of the deployed {best_m} 5-class model on KDDTest+ (counts and row-normalised).")}
{img(M['figures']['confusion_binary'], f"Confusion matrices of the deployed {best_b} binary model on KDDTest+ at threshold {thr['value']:.3f}.")}"""

    shap_sec = f"""
<p>TreeExplainer SHAP values of the <b>{M['shap']['model']}</b> binary model on {M['shap']['n_rows']:,} random validation rows
(computed in {M['shap']['time_s']:.1f} s). Each value is the feature's contribution to the log-odds of "attack".</p>
{table(["Rank", "Feature", "mean |SHAP|"], [[i + 1, r["feature"], f"{r['mean_abs_shap']:.4f}"] for i, r in enumerate(M["shap"]["top15"])], num_from=2)}
<div class="cols">{img(M['figures']['shap_summary'], "SHAP beeswarm: each dot is one connection; colour is the (preprocessed) feature value.")}
{img(M['figures']['feature_importance'], "Top 15 features by mean absolute SHAP value.")}</div>"""

    sens_tbl = table(["Operating point", "Threshold", "Val recall", "Val FAR", "KDDTest+ recall", "KDDTest+ FAR", "KDDTest+ F1 macro"],
                     [[r["label"], f"{r['threshold']:.3g}", pct(r["val_recall"]), pct(r["val_far"]), pct(r["test_recall"]),
                       pct(r["test_far"]), f4(r["test_f1_macro"])] for r in M["threshold_sensitivity"]], 0)
    thr_sec = f"""
<p>The operating point was chosen on the validation split only: <b>maximise attack recall subject to a false-alarm
rate ≤ {pct(thr['max_far_constraint'], 0)}</b>. Recall can only fall as the threshold rises, so the optimum is the lowest
threshold whose validation FAR still respects the budget. Because the validation split is almost perfectly separable,
this rule spends the whole false-alarm budget on sensitivity – deliberately, since the detector must face attack types
that never occur in validation. Result: threshold <b>{thr['value']:.3g}</b> on P(attack) → validation recall
<b>{pct(thr['val_recall'])}</b>, validation false-alarm rate <b>{pct(thr['val_false_alarm_rate'])}</b>. On KDDTest+ the
same threshold gives recall <b>{pct(dep_b['recall'])}</b> and false-alarm rate <b>{pct(dep_b['false_alarm_rate'])}</b>
({M['error_analysis']['binary_false_alarms']:,} of {A['binary_distribution']['test']['normal']:,} normal connections flagged).</p>
{img(M['figures']['threshold_curve'], "Recall and false-alarm rate as a function of the decision threshold on the validation split (log-scale threshold).")}
<h3>Threshold sensitivity (analysis only – the deployed threshold was fixed before KDDTest+ was read)</h3>
{sens_tbl}
<p>Validation cannot distinguish thresholds between 10<sup>-6</sup> and 0.5 (recall and FAR barely move), whereas on
KDDTest+ the choice moves recall by tens of points: unseen attacks receive low but non-zero attack probabilities, so a
sensitive threshold catches many of them at the cost of a higher false-alarm rate under distribution shift.</p>"""

    ea = M["error_analysis"]
    err_sec = f"""
<p>Deployed models on KDDTest+: <b>{ea['binary_total_missed_attacks']:,}</b> of {total_test_attacks:,} attacks missed by the binary model,
<b>{ea['binary_false_alarms']:,}</b> false alarms.</p>
<h3>Attack types most often missed (binary)</h3>
{table(["Attack", "Class", "Test rows", "Missed", "Miss rate", "Seen in train"],
       [[r["attack"], r["class"], f"{r['n']:,}", f"{r['binary_missed']:,}", pct(r["binary_miss_rate"], 1), "yes" if r["seen_in_train"] else "<b>no</b>"] for r in ea["most_missed_binary"]], num_from=2)}
<h3>Attack types most often mis-classified (5-class)</h3>
{table(["Attack", "True class", "Test rows", "Wrong", "Error rate", "Most common prediction", "Seen in train"],
       [[r["attack"], r["class"], f"{r['n']:,}", f"{r['multiclass_wrong']:,}", pct(r["multiclass_error_rate"], 1), r["multiclass_most_common_prediction"], "yes" if r["seen_in_train"] else "<b>no</b>"] for r in ea["most_misclassified_multiclass"]], num_from=2)}"""

    limits_sec = """
<ul>
<li><b>Dataset age and realism.</b> NSL-KDD derives from 1998/99 DARPA traffic; services, attack tooling and traffic mixes have changed. Numbers here measure method quality, not expected performance on a modern network.</li>
<li><b>Unseen attacks.</b> Supervised classifiers can only generalise from feature similarity; zero-day detection needs anomaly / novelty detection (e.g. isolation forest or autoencoder on normal traffic) alongside the classifier.</li>
<li><b>R2L / U2R.</b> With few hundred training rows and strong distribution shift, per-class recall is low. Options: targeted oversampling or focal loss, hierarchical models (attack → family), or cost-sensitive thresholds per class.</li>
<li><b>Thresholding.</b> The threshold maximises recall under a 5 % validation FAR; because validation is in-distribution, the test FAR differs. Calibrating on a small labelled sample of target traffic would tighten this.</li>
<li><b>Tuning budget.</b> 12 random configurations × 3 folds per booster to stay within the 25-minute CPU budget; a larger search or early-stopping tuning may add a little macro-F1 on Protocol A but is unlikely to close the Protocol B gap.</li>
<li><b>Next steps.</b> Evaluate on CIC-IDS2017 / UNSW-NB15, add an anomaly-score channel, estimate calibration (Brier / reliability), and package the pipeline as a streaming scorer with drift monitoring on the top SHAP features.</li>
</ul>"""

    sections = [
        ("problem", "1. Problem", f"""<p><b>IntruShield</b> is a network intrusion detection model that classifies individual connection records as
<i>normal</i> or <i>attack</i> (binary task) and assigns attacks to one of four families – DoS, Probe, R2L, U2R
(5-class task). Operational priorities: high detection rate, a bounded false-alarm rate (≤ 5 % on normal traffic), and
honest measurement of how performance degrades on attack types never seen in training.</p>{kpis}"""),
        ("data", "2. Dataset &amp; Audit", audit_sec),
        ("prep", "3. Preprocessing", prep_sec),
        ("models", "4. Models Compared", models_sec),
        ("results", "5. Results – Protocol A vs Protocol B", results_sec),
        ("gap", "6. Why the gap exists – unseen attacks", gap_sec),
        ("perclass", "7. Per-class Performance", perclass_sec),
        ("shap", "8. Explainability (SHAP)", shap_sec),
        ("threshold", "9. Threshold &amp; False Alarms", thr_sec),
        ("errors", "10. Error Analysis", err_sec),
        ("limits", "11. Limitations &amp; Next Steps", limits_sec),
    ]
    nav = "".join(f'<a href="#{i}">{t.split(". ", 1)[1].replace("&amp;", "&")}</a>' for i, t, _ in sections)
    body = "".join(f'<section id="{i}"><h2>{t}</h2>{h}</section>' for i, t, h in sections)
    html = f"""<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>IntruShield – NSL-KDD Intrusion Detection Evaluation</title><style>{CSS}</style></head><body>
<header><h1>IntruShield · Network Intrusion Detection on NSL-KDD</h1>
<p>Evaluation report · generated {datetime.now():%Y-%m-%d %H:%M} · binary model: {best_b} · 5-class model: {best_m} · random_state=42</p></header>
<nav>{nav}</nav><main>{body}</main>
<footer>All figures and numbers in this report are produced by the IntruShield pipeline (src/data.py → train.py → evaluate.py → report.py). No external benchmarks are quoted.</footer>
</body></html>"""
    out = REPORTS_DIR / "eval_report.html"
    out.write_text(html, encoding="utf-8")
    print(f"Report written to {out} ({out.stat().st_size / 1e6:.1f} MB)")


if __name__ == "__main__":
    main()

In [ ]:
%run src/report.py

## 8. Live demo — `predict.py`
Loads the saved models, scores 10 random KDDTest+ connections and explains each decision with its top-3 SHAP reasons.

In [ ]:
%%writefile src/predict.py
"""IntruShield - demo: score 10 random KDDTest+ connections with the saved models + SHAP reasons."""
import warnings

import joblib
import numpy as np
import shap

from config import FEATURES, MODELS_DIR, RANDOM_STATE
from data import load_split

warnings.filterwarnings("ignore")


def shap_reasons(pipe, X, top=3):
    """Top-k signed SHAP contributions (log-odds) toward 'attack' for each row of X."""
    clf, prep = pipe["clf"], pipe["prep"]
    Xt = prep.transform(X)
    names = prep.get_feature_names_out()
    if hasattr(clf, "coef_"):  # linear model: contribution = coef * standardized value
        sv = Xt * clf.coef_[0]
    else:
        sv = shap.TreeExplainer(clf).shap_values(Xt)
        if isinstance(sv, list):
            sv = sv[1]
        elif sv.ndim == 3:
            sv = sv[:, :, 1]
    out = []
    for i in range(len(X)):
        order = np.argsort(-np.abs(sv[i]))[:top]
        out.append([(names[j], float(Xt[i, j]), float(sv[i, j])) for j in order])
    return out


def main(n=10):
    binary = joblib.load(MODELS_DIR / "binary_model.joblib")
    multi = joblib.load(MODELS_DIR / "multiclass_model.joblib")
    test = load_split("test")
    rows = test.sample(n=n, random_state=RANDOM_STATE)
    X = rows[FEATURES]
    p_attack = binary["pipeline"].predict_proba(X)[:, 1]
    is_attack = p_attack >= binary["threshold"]
    cls = multi["label_encoder"].inverse_transform(multi["pipeline"].predict(X))
    reasons = shap_reasons(binary["pipeline"], X)

    print(f"IntruShield demo - binary={binary['model_name']} (threshold {binary['threshold']:.3g}), "
          f"multiclass={multi['model_name']}\n")
    hdr = f"{'#':>2} {'true attack':<16}{'true class':<11}{'pred':<8}{'pred class':<11}{'P(attack)':>9}  top-3 SHAP reasons"
    print(hdr)
    print("-" * len(hdr))
    for k, (idx, r) in enumerate(rows.iterrows()):
        pred = "ATTACK" if is_attack[k] else "normal"
        pred_cls = cls[k] if is_attack[k] else "Normal"
        why = "; ".join(f"{f}={v:.2f} ({s:+.2f})" for f, v, s in reasons[k])
        flag = "" if (pred_cls == r['label5']) else "  <-- wrong"
        print(f"{k + 1:>2} {r['attack']:<16}{r['label5']:<11}{pred:<8}{pred_cls:<11}{p_attack[k]:>9.4f}  {why}{flag}")
    print("\nSHAP value = contribution to log-odds of 'attack' (+ pushes toward attack); feature values "
          "shown after log1p / one-hot preprocessing.")


if __name__ == "__main__":
    main()

In [ ]:
%run src/predict.py

## 9. One-command runner — `run_all.py`
Runs every step in order and prints the results summary. Outside the notebook: `python run_all.py` (or `run_all.bat`).

In [ ]:
%%writefile run_all.py
"""IntruShield - run the whole pipeline: data audit -> train -> evaluate -> report -> demo."""
import json
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parent
PY = sys.executable
STEPS = ["data.py", "train.py", "evaluate.py", "report.py", "predict.py"]
MODEL_ORDER = ["LogisticRegression", "RandomForest", "XGBoost", "LightGBM"]


def main():
    t0 = time.perf_counter()
    timings = {}
    for step in STEPS:
        print(f"\n{'=' * 78}\n>>> python src/{step}\n{'=' * 78}", flush=True)
        t = time.perf_counter()
        rc = subprocess.call([PY, str(ROOT / "src" / step)], cwd=ROOT)
        timings[step] = time.perf_counter() - t
        if rc != 0:
            sys.exit(f"\nStep {step} failed with exit code {rc}. Aborting.")
    total = time.perf_counter() - t0

    M = json.loads((ROOT / "reports" / "metrics.json").read_text())
    P = M["protocols"]
    print(f"\n{'=' * 78}\nIntruShield - RESULTS SUMMARY\n{'=' * 78}")
    for task in ("binary", "multiclass"):
        print(f"\n{task.upper():<10}  {'Protocol A (validation)':^34} | {'Protocol B (KDDTest+)':^34} | B-21")
        print(f"{'model':<20}{'F1m':>8}{'Acc':>8}{'Recall':>8}{'FAR':>8} |{'F1m':>8}{'Acc':>8}{'Recall':>8}{'FAR':>8} |{'F1m':>8}")
        for n in MODEL_ORDER:
            a, b, c = P["A: validation"][task][n], P["B: KDDTest+"][task][n], P["B: KDDTest-21"][task][n]
            star = "*" if n == M["best"][task] else " "
            print(f"{star}{n:<19}{a['f1_macro']:>8.4f}{a['accuracy']:>8.4f}{a['recall']:>8.4f}{a['false_alarm_rate']:>8.4f} |"
                  f"{b['f1_macro']:>8.4f}{b['accuracy']:>8.4f}{b['recall']:>8.4f}{b['false_alarm_rate']:>8.4f} |{c['f1_macro']:>8.4f}")
    print("\n* = selected on validation macro-F1.  FAR = false-alarm rate on normal traffic.")
    db = M["deployed"]["binary"]["B: KDDTest+"]
    svu = M["seen_vs_unseen"]["deployed"]
    print(f"Deployed binary {M['best']['binary']} (thr {M['threshold']['value']:.3f}) on KDDTest+: recall {db['recall']:.4f}, "
          f"FAR {db['false_alarm_rate']:.4f}, AUC {db['roc_auc']:.4f} | seen-attack recall {svu['seen']['recall']:.3f}, "
          f"unseen-attack recall {svu['unseen']['recall']:.3f}")
    lk = M["leakage_check"]
    if lk["triggered"]:
        print(f"Leakage check: Protocol A accuracy {lk['max_protocol_a_accuracy']:.4%} > 99.9%; "
              f"{lk['val_rows_with_identical_feature_vector_in_train_split']} / {lk['val_rows']} validation rows duplicate a train row.")
    print("\nStep timings: " + ", ".join(f"{k} {v:.0f}s" for k, v in timings.items()) + f"  | TOTAL {total / 60:.1f} min")
    print(f"\nReport: {ROOT / 'reports' / 'eval_report.html'}")


if __name__ == "__main__":
    main()

In [ ]:
# Uncomment to re-run the entire pipeline from scratch in one go (≈ 8–10 min):
# %run run_all.py

## 10. Key takeaways
- **Two protocols, honest numbers.** Near-perfect scores on the in-distribution validation split reflect how easy
  NSL-KDD is within one distribution (the leakage check finds only a handful of duplicate rows). KDDTest+ is the real test.
- **Generalisation to new attacks.** The seen-vs-unseen split shows how much of the drop comes from the 17 attack
  types that never appear in training.
- **Trade-off, not a single number.** Random Forest catches more attacks on KDDTest+ but raises far more false alarms;
  the deployed model is chosen on validation, never on test.
- **Weak spot:** R2L and U2R — rare in training and mostly predicted as *Normal*, because password guessing or SNMP
  queries look like legitimate traffic at the connection level.
- **Next steps:** anomaly detection for zero-days, oversampling / hierarchical models for R2L/U2R, and validation on
  a modern dataset (CIC-IDS2017 / UNSW-NB15).